# 11 - Amendment 2B-C - group spectra and the montage error (descriptive)

Two descriptive figures of the harmonised data, decided after the results were seen (post hoc amendment 2,
config/post_hoc_plan_2.json, frozen before anything is computed):

- **Group spectra (Fig. S2).** The EEG files are read again with exactly the preprocessing of the features
  (features.preprocessed_epochs: 19 channels, average reference, 40 Hz low-pass of the Greek data, 0.5–45 Hz FIR,
  4-s epochs, RMS scaling) and the same Welch settings. Each epoch's spectrum is averaged over the 19 channels and
  divided by its power in 0.5-45 Hz, then averaged over the participant's epochs. Only the participants with
  analysable EEG are used. Group curves (mean and bootstrap 95% band) go to outputs/descriptive/spectra_by_group.csv,
  which holds no participant-level data; participant spectra stay in outputs/descriptive/.
- **Montage error (Fig. S1).** The electrode positions used by notebook 01 for the BrainLat 10-20 correspondence,
  checked against Table S1, saved for the head map.

No test, no key number; notebook 08 is unaffected. The figures are drawn by notebook 12.

**Time:** reading the files again, roughly as long as the feature extraction of notebook 01 without the
connectivity features (on the full data, typically 5–20 minutes). Cached: a second run takes seconds.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from eegad.config import load_context
from eegad.data import load_table
from eegad.features import brainlat_channel_map
from eegad.posthoc import POST_HOC_PLAN_2, freeze_post_hoc_plan
from eegad.reporting import write_json
from eegad.spectra import cohort_spectra, group_spectra, montage_geometry

ctx = load_context("11")
out, plan = ctx.out, ctx.plan
PH, PH_HASH, PH_FROZEN = freeze_post_hoc_plan(ctx, POST_HOC_PLAN_2)
DESC = out / "descriptive"
DESC.mkdir(exist_ok=True)
subj_bl, subj_gr = load_table(out / "derived" / "brainlat_subjects.csv"), load_table(out / "derived" / "greek_subjects.csv")
s1 = pd.read_csv(out / "tables" / "S1_brainlat_channel_map_1020.csv")
CHANNELS = ctx.channels

## Group spectra

In [ ]:
map_bl = dict(zip(s1["channel_brainlat"], s1["channel_1020"]))
map_gr = {c: c for c in CHANNELS}
base = {"channels": CHANNELS, "prep": plan["preprocessing"], "max_epochs": ctx.rt["max_epochs"], "version": 1}
items_bl = [(r.subject_id, r.file, map_bl) for r in subj_bl.itertuples()]
items_gr = [(r.subject_id, r.file, map_gr) for r in subj_gr.itertuples()]
spec_bl = cohort_spectra(items_bl, f"brainlat_{ctx.tag}", {**base, "mapping": map_bl,
                                                           "files": [[s, str(f)] for s, f, _ in items_bl]}, ctx)
spec_gr = cohort_spectra(items_gr, f"greek_{ctx.tag}", {**base, "mapping": map_gr, "band_harmonization": True,
                                                        "files": [[s, str(f)] for s, f, _ in items_gr]},
                         ctx, harmonize_band=True)
for name, spec, subj in (("BrainLat", spec_bl, subj_bl), ("Greek", spec_gr, subj_gr)):
    missing = sorted(set(subj.subject_id) - set(spec.subject_id))
    print(f"{name}: spectra of {len(spec)} of {len(subj)} participants" + (f" (missing: {missing})" if missing else ""))
GROUPS = pd.concat([group_spectra(spec_bl, subj_bl, "BrainLat", ctx.n_boot, ctx.seed),
                    group_spectra(spec_gr, subj_gr, "Greek", ctx.n_boot, ctx.seed)], ignore_index=True)
GROUPS.to_csv(DESC / "spectra_by_group.csv", index=False)
peak = (GROUPS[(GROUPS.frequency_hz >= 6) & (GROUPS.frequency_hz <= 14)]
        .sort_values("mean").groupby(["cohort", "group"]).tail(1)[["cohort", "group", "n", "frequency_hz"]])
print("Frequency of the highest relative density between 6 and 14 Hz (descriptive):")
display(peak.rename(columns={"frequency_hz": "peak (Hz)"}).reset_index(drop=True))

## Montage geometry (the positions behind Table S1)

In [ ]:
bl_files = load_table(out / "derived" / "brainlat_files.csv")
MAP, source, montage_name, file0, _ = brainlat_channel_map(bl_files["file"], CHANNELS)
ELEC, TARG, SPACING = montage_geometry(file0, CHANNELS, s1, source)
ELEC.to_csv(DESC / "montage_electrodes.csv", index=False)
TARG.to_csv(DESC / "montage_targets.csv", index=False)
write_json(DESC / "montage_spacing.json", SPACING)
print(f"Source of the positions: {source} | 10-20 reference: {montage_name}")
print(f"Angle to the electrode used: mean {TARG.angle_deg.mean():.1f}°, maximum {TARG.angle_deg.max():.1f}° | "
      f"nearest 10-20 neighbours: median {SPACING['median_neighbour_angle_deg']:.1f}°, "
      f"minimum {SPACING['min_neighbour_angle_deg']:.1f}°")
display(TARG[["channel_1020", "angle_deg", "nearest_1020_neighbour", "neighbour_angle_deg"]].round(1))
write_json(DESC / "descriptive_amendment.json", {"amendment": PH["amendment"], "post_hoc_plan_hash": PH_HASH,
                                                 "frozen_at": PH_FROZEN, "mode": ctx.tag})